# Explorando los datos del Expense Validator

**Cómo usarla:** ejecuta una celda a la vez con `Shift + Enter`. Cada sección tiene una pregunta; cambia el código y vuelve a ejecutar para jugar con los datos.

Reglas del juego:
- Las celdas que dicen **ENTRADA** usan solo lo que el sistema puede ver.
- Las celdas que dicen **RESPUESTAS** usan el answer key (`labels`, `notes`). Es para que tú entiendas y califiques, nunca para el pipeline.

In [ ]:
import sys, json, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200, "display.max_colwidth", 60, "display.max_rows", 100)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from validator.loader import load_receipts            # ENTRADA: sin respuestas
from validator.gold import load_gold, OUT_OF_SCOPE    # RESPUESTAS: solo para calificar

receipts = load_receipts()
gold = load_gold()
by_id = {r["receipt_id"]: r for r in receipts}
scored = [rid for rid in gold if rid not in OUT_OF_SCOPE]
print(len(receipts), "recibos,", sum(len(r["line_items"]) for r in receipts), "líneas;", len(scored), "se califican")

## 1 · Un recibo completo (ENTRADA)

Esto es lo que ve el sistema para RCP-010. Fíjate en dos bloques: el **contexto** (tipo declarado, MCC, asistentes, viaje, aprobación) y las **líneas**.

In [ ]:
print(json.dumps(by_id["RCP-010"], indent=2, ensure_ascii=False))

Una vista más compacta que vamos a usar mucho. Cambia el ID por cualquier otro (`RCP-001` … `RCP-032`).

In [ ]:
def show(rid):
    r = by_id[rid]
    print(f"{rid} | {r['submitted_expense_type']} | {r['merchant_name']} (MCC {r['mcc']}) | {r['entity']} {r['employee_country']}")
    print(f"asistentes={r['attendee_count']}  viaje={r['trip_id']} ({r['trip_nights']} noches)  ref={r['pre_approval_ref']}  itemizado={r['itemized']}")
    for li in r["line_items"]:
        print(f"  {li['line_id'][-2:]}  {li['description_raw']!r:62} {li['quantity']} x {li['unit_price']} = {li['amount']} {r['currency']}  (${li['amount_usd']})")
    print(f"  total {r['total_amount']} {r['currency']} = ${r['total_amount_usd']}")

show("RCP-010")

## 2 · Los 32 recibos en una tabla (ENTRADA)

Una fila por recibo. **Preguntas para jugar:** ¿cuántos no tienen asistentes? ¿cuáles no son de EE. UU.? ¿cuál no está desglosado?

In [ ]:
df_r = pd.DataFrame([dict(
    id=r["receipt_id"], employee=r["employee_id"], country=r["employee_country"], entity=r["entity"],
    type=r["submitted_expense_type"], merchant=r["merchant_name"], mcc=r["mcc"], currency=r["currency"],
    total_usd=r["total_amount_usd"], attendees=r["attendee_count"], trip=r["trip_id"], nights=r["trip_nights"],
    ref=r["pre_approval_ref"], itemized=r["itemized"], n_lines=len(r["line_items"])) for r in receipts]).set_index("id")
df_r

In [ ]:
# Prueba: recibos sin número de asistentes
df_r[df_r.attendees.isna()]

In [ ]:
# Prueba: recibos de otras regiones, no itemizados, con pre-aprobación...
display(df_r[df_r.entity != "GLOBAL"])
display(df_r[~df_r.itemized])
display(df_r[df_r.ref.notna()])

## 3 · Las 79 líneas (ENTRADA)

**Pregunta:** ¿cuántas líneas no están en inglés? Esas son las que necesitan traducción.

In [ ]:
df_l = pd.DataFrame([dict(
    receipt=r["receipt_id"], line=li["line_id"], lang=li["language"], text=li["description_raw"],
    qty=li["quantity"], unit_price=li["unit_price"], amount=li["amount"], amount_usd=li["amount_usd"])
    for r in receipts for li in r["line_items"]])
print(df_l.lang.value_counts().to_string())
df_l[df_l.lang != "en"]

## 4 · ¿Cuadra la aritmética? (ENTRADA)

Verificamos que cantidad × precio = monto, que las líneas suman el total y que el tipo de cambio convierte bien.

In [ ]:
bad = []
for r in receipts:
    s = sum(li["amount"] for li in r["line_items"])
    if abs(s - r["total_amount"]) > 0.011: bad.append((r["receipt_id"], "suma de líneas", s, r["total_amount"]))
    if abs(r["total_amount"] * r["fx_rate_to_usd"] - r["total_amount_usd"]) > 0.011: bad.append((r["receipt_id"], "fx", r["total_amount"] * r["fx_rate_to_usd"], r["total_amount_usd"]))
    for li in r["line_items"]:
        if abs(li["quantity"] * li["unit_price"] - li["amount"]) > 0.011: bad.append((li["line_id"], "qty*precio", li["quantity"] * li["unit_price"], li["amount"]))
bad or "Todo cuadra"

## 5 · Las respuestas: veredictos y categorías (RESPUESTAS)

Aquí ya miramos el answer key. Debe salir 9 pass, 16 flag y 3 review en los 28 que se califican.

In [ ]:
disp = pd.Series([gold[r]["labels"]["disposition"] for r in scored])
print(disp.value_counts().to_string())
disp.value_counts().plot.bar(color=["#c23a3a", "#2f7d46", "#b7791f"], figsize=(4, 3)); plt.title("Veredicto correcto (28 recibos)"); plt.show()

In [ ]:
cats = pd.Series([lab["category"] for rid in scored for lab in gold[rid]["line_labels"].values()]).value_counts()
print(len(cats), "categorías distintas en", cats.sum(), "líneas")
ax = cats.plot.barh(figsize=(7, 6)); ax.invert_yaxis(); plt.title("Categoría correcta de las 65 líneas"); plt.show()

In [ ]:
# ¿Qué reglas aparecen en las respuestas, y cuántas veces?
rules = pd.Series([e["rule_id"] for rid in scored for e in gold[rid]["labels"]["expected_exceptions"]]).value_counts()
sigs = pd.Series([s["rule_id"] for rid in scored for s in gold[rid]["labels"]["review_signals"]]).value_counts()
display(rules.to_frame("excepciones")); display(sigs.to_frame("señales"))

## 6 · El mismo producto, distinto contexto (ENTRADA + RESPUESTAS)

Todos los recibos que tienen una línea de **alcohol**. Mira cómo el tipo declarado, los asistentes, el MCC y la región cambian la regla y el veredicto. Esta tabla es la idea central del proyecto.

In [ ]:
rows = []
for rid, g in gold.items():
    if any(l["category"] == "alcohol" for l in g["line_labels"].values()):
        r = by_id[rid]
        rows.append(dict(id=rid, type=r["submitted_expense_type"], attendees=r["attendee_count"], mcc=r["mcc"], entity=r["entity"],
                         alcohol_rules=[e["rule_id"] for e in g["labels"]["expected_exceptions"] if e["rule_id"].startswith(("ALC", "REG-01"))],
                         verdict=g["labels"]["disposition"]))
pd.DataFrame(rows).set_index("id")

## 7 · ¿El monto predice el veredicto? (ENTRADA + RESPUESTAS)

In [ ]:
colors = {"pass": "#2f7d46", "flag": "#c23a3a", "review": "#b7791f"}
fig, ax = plt.subplots(figsize=(8, 3))
for rid in scored:
    v = gold[rid]["labels"]["disposition"]
    ax.scatter(by_id[rid]["total_amount_usd"], v, color=colors[v], s=60, alpha=.8)
ax.set_xlabel("total en USD"); ax.set_title("Cada punto es un recibo"); plt.show()

## 8 · Leer una regla por su ID

Esta función busca el texto de una regla en los documentos de política. Prueba con `ALC-01`, `TECH-02`, `PERS-02`, `EXC-02`, `MEAL-01`, `GIFT-04`…

In [ ]:
def show_rule(rid):
    for path in sorted((ROOT / "data" / "policy-docs").glob("*.md")):
        lines = path.read_text().splitlines()
        for i, line in enumerate(lines):
            if line.startswith(f"**{rid}"):
                j = i + 1
                while j < len(lines) and not re.match(r"\*\*[A-Z]+-\d+|#", lines[j]):
                    j += 1
                print(f"[{path.name}]\n" + "\n".join(lines[i:j]).strip())
                return
    print("no encontrada")

show_rule("ALC-01")

In [ ]:
show_rule("TECH-02")

## 9 · Calculadora del tope de comida (MEAL-01)

Gasto de comida = líneas `food_beverage` + `alcohol` + `service_fee`, dividido entre los asistentes (o 1 si no hay). Si no está desglosado, cuenta el total. Esta celda usa las categorías **correctas** para que veas la regla sola.

In [ ]:
def meal_per_person(rid):
    r, g = by_id[rid], gold[rid]["line_labels"]
    spend = r["total_amount_usd"] if not r["itemized"] else sum(
        li["amount_usd"] for li in r["line_items"] if g[li["line_id"]]["category"] in ("food_beverage", "alcohol", "service_fee"))
    people = r["attendee_count"] or 1
    return dict(id=rid, spend=round(spend, 2), people=people, per_person=round(spend / people, 2), cap=75 if r["currency"] == "USD" else "ver REG-02/03")

pd.DataFrame([meal_per_person(x) for x in ["RCP-002", "RCP-007", "RCP-015", "RCP-023"]]).set_index("id")

## 10 · Juega a ser el motor: predice y compara

1. Ejecuta `show(...)` y piensa: ¿pass, review o flag? ¿Qué regla?
2. Solo después ejecuta `reveal(...)`.

Orden sugerido (de fácil a trampa): `RCP-001`, `RCP-010`, `RCP-026`, `RCP-016`, `RCP-014`, `RCP-022`, `RCP-028`.

In [ ]:
def reveal(rid):
    g = gold[rid]
    print("VEREDICTO:", g["labels"]["disposition"])
    for e in g["labels"]["expected_exceptions"]:
        print(f"  excepción {e['rule_id']} ({e['severity']}) líneas={[x[-2:] for x in e['line_ids']]}\n    {e['reason']}")
    for s in g["labels"]["review_signals"]:
        print(f"  señal {s['rule_id']}: {s['reason']}")
    print("NOTA:", g["notes"])

show("RCP-026")

In [ ]:
reveal("RCP-026")

In [ ]:
show("RCP-016")

In [ ]:
reveal("RCP-016")